# Half-year ACTIVSg2000 DCOPF benchmark

Run this notebook to generate, rather than assert, measurements for horizons 1, 24, 168, 720, 4368, and 4416 hours. No benchmark results are embedded.

## Setup

This installs the declared project dependencies and reporting helpers. The explicit highspy guard is required: if HiGHS cannot be imported, the notebook stops with an actionable error.

In [ ]:
import importlib.metadata
import os
from pathlib import Path
import platform
import subprocess
import sys

REPO_URL = os.environ.get("REPO_URL", "https://github.com/MhaoMou/ercot-connection-planning-instructor.git")
REPO_REF = os.environ.get("REPO_REF", "main")
REPO_DIR = Path(os.environ.get("REPO_DIR", "/content/ercot-connection-planning-instructor"))

if not (REPO_DIR / "pyproject.toml").is_file():
    if REPO_DIR.exists():
        raise RuntimeError(f"{REPO_DIR} exists but is not the expected repository")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "psutil", "pandas", "matplotlib"], check=True)
try:
    import highspy
except ImportError as exc:
    raise RuntimeError(
        "highspy is unavailable. Install the project dependencies in this cell and rerun; "
        "the benchmark cannot produce solver measurements without HiGHS."
    ) from exc

REPO_REVISION = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("repository:", REPO_DIR)
print("revision:", REPO_REVISION)
print("python:", sys.version)
print("platform:", platform.platform())
print("highspy:", importlib.metadata.version("highspy"))

## Configuration and prepared solver inputs

The checked-in scenario manifests are fail-closed until release validation is complete. This notebook therefore loads each checked-in prepared artifact with `load_prepared_artifact`, reconstructs its deterministic demand/network inputs, and calls the same public low-level DCOPF model with source status, PMIN, and nameplate PMAX limits. No synthetic hourly generator availability caps are applied. The exact benchmark input path is `load_prepared_artifact -> static_inputs -> period_inputs -> solve_dcopf / solve_dcopf_horizon`.


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from functools import lru_cache
import csv
import json
import math
import os
import statistics
import threading
import time
import tracemalloc

import pandas as pd
import psutil

from tools.activsg2000 import dcopf as dcopf_module
from tools.activsg2000.dcopf import (
    Branch, Bus, Generator, StorageDispatchAsset,
    load_prepared_artifact, solve_dcopf, solve_dcopf_horizon,
)
from ercot_game.scenario_profiles import block_offset, demand_multiplier, load_profile_inputs

BENCHMARK_HORIZONS = (1, 24, 168, 720, 4368, 4416)
SHORT_HORIZON_CUTOFF = 720
REPEATS_SHORT = 3
REPEATS_LONG = 1
_storage_cap_text = os.environ.get("STORAGE_MAX_HORIZON")
STORAGE_MAX_HORIZON = None
if _storage_cap_text is not None:
    STORAGE_MAX_HORIZON = int(_storage_cap_text)
    if STORAGE_MAX_HORIZON <= 0:
        raise ValueError("STORAGE_MAX_HORIZON must be a positive integer when explicitly set")
PARALLEL_WORKERS = max(2, min(4, os.cpu_count() or 2))
WALL_TIME_BUDGET_S = float(os.environ.get("WALL_TIME_BUDGET_S", "1800"))
PEAK_RSS_BUDGET_MB = float(os.environ.get("PEAK_RSS_BUDGET_MB", "12000"))
RSS_SAMPLING_INTERVAL_S = float(os.environ.get("RSS_SAMPLING_INTERVAL_S", "0.05"))
if RSS_SAMPLING_INTERVAL_S <= 0:
    raise ValueError("RSS_SAMPLING_INTERVAL_S must be positive")
SCENARIO_ROOT = Path(os.environ.get(
    "ARTIFACT_ROOT", str(REPO_DIR / "data/processed/activsg2000_scenarios_v1")
))
ARTIFACT_4368 = SCENARIO_ROOT / "2028/jan_jun"
ARTIFACT_4416 = SCENARIO_ROOT / "2027/jul_dec"
PREPARED_SOLVER_PATH = (
    "load_prepared_artifact -> static_inputs -> period_inputs -> "
    "solve_dcopf / solve_dcopf_horizon"
)
GENERATOR_LIMIT_SEMANTICS = (
    "prepared source status, PMIN, and nameplate PMAX; no synthetic hourly caps"
)
CHECKPOINT_DIR = Path(os.environ.get(
    "CHECKPOINT_DIR", str(REPO_DIR / "experiments/colab/results")
))
CHECKPOINT_JSON = CHECKPOINT_DIR / "dcopf_benchmark_results.json"
CHECKPOINT_CSV = CHECKPOINT_DIR / "dcopf_benchmark_results.csv"


def artifact_for(horizon):
    if horizon <= 4368:
        return ARTIFACT_4368
    if horizon <= 4416:
        return ARTIFACT_4416
    raise ValueError(f"unsupported horizon: {horizon}")


@lru_cache(maxsize=4)
def static_inputs(artifact_dir_string):
    root = Path(artifact_dir_string)
    artifact = load_prepared_artifact(root)
    if artifact.scenario_year is None or artifact.scenario_block is None:
        raise ValueError(f"{root} is missing scenario_profile.json")
    base_load, _profiles, _catalog = load_profile_inputs(root)
    generator_rows = tuple(
        row for row in artifact.generators_raw if row.get("status", "1") == "1"
    )
    branch_rows = tuple(
        row for row in artifact.branches_raw
        if row.get("status", "1") == "1" and float(row["reactance_pu"]) != 0.0
    )
    injections = {}
    for row in artifact.dcline_raw:
        if row.get("status", "1") == "1":
            injections[row["to_bus_id"]] = injections.get(row["to_bus_id"], 0.0) + float(row["Pt"])
    return artifact, base_load, generator_rows, branch_rows, injections


@lru_cache(maxsize=64)
def period_inputs(artifact_dir_string, period_index):
    artifact, base_load, generator_rows, branch_rows, injections = static_inputs(artifact_dir_string)
    absolute_hour = block_offset(artifact.scenario_year, artifact.scenario_block) + period_index
    multiplier = demand_multiplier(absolute_hour, year=artifact.scenario_year)
    demand = {bus_id: round(value * multiplier, 8) for bus_id, value in base_load.items()}
    buses = tuple(
        Bus(row["bus_id"], demand[row["bus_id"]], reference=row.get("type") == "3")
        for row in artifact.buses_raw
    )
    generators = tuple(
        Generator(
            row["generator_id"], row["bus_id"], float(row["p_min_mw"]),
            float(row["p_max_mw"]), float(row["marginal_cost_c1_per_mwh"])
        )
        for row in generator_rows
    )
    branches = tuple(
        Branch(
            row["branch_id"], row["from_bus_id"], row["to_bus_id"],
            1.0 / float(row["reactance_pu"]), float(row["thermal_limit_mva"])
        )
        for row in branch_rows
    )
    return buses, generators, branches, injections


def model_counts(artifact_dir, horizon, storage_assets=0, owner_groups=0):
    artifact, _base, generator_rows, branch_rows, _inj = static_inputs(str(artifact_dir))
    buses = len(artifact.buses_raw)
    gens = len(generator_rows)
    lines = len(branch_rows)
    network_vars = horizon * (gens + max(buses - 1, 0))
    return {
        "bus_count": buses, "generator_count": gens, "branch_count": lines,
        "network_variable_count": network_vars,
        "variable_count": network_vars + horizon * 4 * storage_assets,
        "constraint_count": horizon * (buses + lines + 4 * storage_assets + owner_groups),
    }


## Measurement helpers

`serial` clears the artifact/profile/topology caches before each period. `sparse_reuse` warms and reuses them. `chunked_parallel` uses deterministic threaded chunks, matching the existing worker pattern. Storage is one coupled horizon solve, so hourly P95 fields remain null. Every case samples process RSS, which includes native HiGHS allocations; the configured polling interval and the possibility of missing a shorter-lived peak between samples are recorded with each row.


In [ ]:
def clear_caches():
    static_inputs.cache_clear()
    period_inputs.cache_clear()
    dcopf_module.clear_prepared_artifact_cache()
    dcopf_module._cached_network_index.cache_clear()


def percentile(values, p):
    if not values:
        return None
    values = sorted(float(x) for x in values)
    if len(values) == 1:
        return values[0]
    position = (len(values) - 1) * p / 100.0
    lo, hi = math.floor(position), math.ceil(position)
    if lo == hi:
        return values[lo]
    return values[lo] + (values[hi] - values[lo]) * (position - lo)


def solve_one(root, index):
    buses, generators, branches, injections = period_inputs(str(root), index)
    return solve_dcopf(
        buses=buses, generators=generators, branches=branches,
        fixed_injection_mw_by_bus=injections, base_mva=100.0,
    )


def solve_serial(root, horizon, cold):
    if not cold:
        clear_caches()
    times, statuses = [], []
    for index in range(horizon):
        if cold:
            clear_caches()
        started = time.perf_counter()
        result = solve_one(root, index)
        times.append(time.perf_counter() - started)
        statuses.append(result.status)
    return {"hourly_times": times, "statuses": statuses}


def solve_chunked_parallel(root, horizon, workers):
    clear_caches()
    chunk_size = math.ceil(horizon / workers)
    chunks = [range(start, min(start + chunk_size, horizon))
              for start in range(0, horizon, chunk_size)]

    def run_chunk(chunk):
        times, statuses = [], []
        for index in chunk:
            started = time.perf_counter()
            result = solve_one(root, index)
            times.append(time.perf_counter() - started)
            statuses.append(result.status)
        return times, statuses

    with ThreadPoolExecutor(max_workers=workers) as executor:
        pieces = list(executor.map(run_chunk, chunks))
    return {
        "hourly_times": [t for times, _ in pieces for t in times],
        "statuses": [s for _, statuses in pieces for s in statuses],
    }


def solve_storage(root, horizon):
    clear_caches()
    periods, injections, owner_load = [], [], []
    for index in range(horizon):
        buses, generators, branches, fixed = period_inputs(str(root), index)
        periods.append((buses, generators, branches))
        injections.append(fixed)
        owner_load.append({buses[0].bus_id: max(buses[0].demand_mw, 0.0)})
    storage = StorageDispatchAsset(
        storage_id="benchmark-storage", team_id="benchmark-team",
        bus_id=periods[0][0][0].bus_id, power_mw=1.0,
        energy_mwh=4.0, initial_soc_mwh=2.0,
    )
    result = solve_dcopf_horizon(
        periods=tuple(periods), storage_assets=(storage,),
        fixed_injection_mw_by_period=tuple(injections),
        owner_load_mw_by_team={"benchmark-team": tuple(owner_load)},
        time_step_hours=1.0, base_mva=100.0,
    )
    return {"hourly_times": None, "statuses": [result.status]}


def _rss_mb(process):
    return process.memory_info().rss / (1024 * 1024)


def timed(function):
    process = psutil.Process()
    stop_event = threading.Event()
    process_rss_before_mb = _rss_mb(process)
    process_rss_peak_mb = [process_rss_before_mb]

    def sample_rss():
        while not stop_event.wait(RSS_SAMPLING_INTERVAL_S):
            try:
                process_rss_peak_mb[0] = max(process_rss_peak_mb[0], _rss_mb(process))
            except psutil.Error:
                break

    sampler = threading.Thread(target=sample_rss, name="dcopf-rss-sampler", daemon=True)
    sampler.start()
    tracemalloc.start()
    started = time.perf_counter()
    payload, error = None, None
    try:
        payload = function()
    except Exception as exc:
        error = {"type": type(exc).__name__, "message": str(exc)}
    finally:
        stop_event.set()
        sampler.join()
        process_rss_peak_mb[0] = max(process_rss_peak_mb[0], _rss_mb(process))
        wall = time.perf_counter() - started
        _current, peak_python = tracemalloc.get_traced_memory()
        tracemalloc.stop()
    return {
        "payload": payload, "error": error, "wall_time_s": wall,
        "peak_memory_mb": process_rss_peak_mb[0],
        "process_rss_peak_mb": process_rss_peak_mb[0],
        "process_rss_before_mb": process_rss_before_mb,
        "rss_sampling_interval_s": RSS_SAMPLING_INTERVAL_S,
        "rss_sampling_limitation": (
            "RSS is sampled process memory and can miss a shorter-lived native "
            "HiGHS peak between polling intervals."
        ),
        "python_peak_memory_mb": peak_python / (1024 * 1024),
    }


def status_text(statuses):
    unique = sorted(set(statuses))
    return "not_run" if not unique else unique[0] if len(unique) == 1 else "|".join(unique)


## Execute all required cases

The default case matrix includes all six official horizons for the storage variant when `highspy` is available. storage horizons are skipped only by explicit configuration such as `STORAGE_MAX_HORIZON`; every such skip is checkpointed with a reason. The notebook checkpoints JSON and CSV after each case and resumes by case key after interruption.


In [ ]:
def case_key(horizon, variant, repeat_index):
    return f"{horizon}:{variant}:{repeat_index}"


def _configuration():
    return {
        "horizons": list(BENCHMARK_HORIZONS),
        "repeats_short": REPEATS_SHORT,
        "repeats_long": REPEATS_LONG,
        "storage_max_horizon": STORAGE_MAX_HORIZON,
        "parallel_workers": PARALLEL_WORKERS,
        "wall_time_budget_s": WALL_TIME_BUDGET_S,
        "peak_rss_budget_mb": PEAK_RSS_BUDGET_MB,
        "rss_sampling_interval_s": RSS_SAMPLING_INTERVAL_S,
        "prepared_solver_path": PREPARED_SOLVER_PATH,
        "generator_limit_semantics": GENERATOR_LIMIT_SEMANTICS,
    }


def _expected_case_keys():
    return [
        case_key(horizon, variant, repeat_index)
        for horizon in BENCHMARK_HORIZONS
        for variant in ("serial", "sparse_reuse", "chunked_parallel", "storage_horizon")
        for repeat_index in range(
            1, (REPEATS_SHORT if horizon <= SHORT_HORIZON_CUTOFF else REPEATS_LONG) + 1
        )
    ]


def load_checkpoint():
    if CHECKPOINT_JSON.is_file():
        try:
            payload = json.loads(CHECKPOINT_JSON.read_text(encoding="utf-8"))
            rows = payload.get("rows", [])
            if isinstance(rows, list):
                return rows
        except (OSError, json.JSONDecodeError):
            pass
    if CHECKPOINT_CSV.is_file():
        try:
            with CHECKPOINT_CSV.open(newline="", encoding="utf-8") as handle:
                return list(csv.DictReader(handle))
        except OSError:
            pass
    return []


def write_checkpoint(rows, recommendation=None):
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    payload = {
        "repository_revision": REPO_REVISION,
        "python": sys.version,
        "platform": platform.platform(),
        "highspy_version": importlib.metadata.version("highspy"),
        "configuration": _configuration(),
        "rows": rows,
        "recommendation": recommendation,
    }
    json_tmp = CHECKPOINT_JSON.with_name(f".{CHECKPOINT_JSON.name}.tmp")
    csv_tmp = CHECKPOINT_CSV.with_name(f".{CHECKPOINT_CSV.name}.tmp")
    json_tmp.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    pd.DataFrame(rows).to_csv(csv_tmp, index=False)
    os.replace(json_tmp, CHECKPOINT_JSON)
    os.replace(csv_tmp, CHECKPOINT_CSV)


def row_for(horizon, root, variant, repeat_index, workers, counts, measurement, storage):
    payload = measurement["payload"] or {}
    error = measurement["error"]
    statuses = payload.get("statuses", [])
    run_status = "completed" if error is None else (
        "infeasible" if "Infeasible" in error["type"] else "error"
    )
    times = payload.get("hourly_times")
    return {
        "horizon_hours": horizon, "artifact_dir": str(root), "variant": variant,
        "storage_variant": "storage" if storage else "no_storage",
        "repeat_index": repeat_index, "worker_count": workers,
        "run_status": run_status, "solver_status": status_text(statuses),
        "wall_time_s": measurement["wall_time_s"],
        "median_hourly_solve_time_s": statistics.median(times) if times else None,
        "p95_hourly_solve_time_s": percentile(times, 95) if times else None,
        "peak_memory_mb": measurement["peak_memory_mb"],
        "process_rss_peak_mb": measurement["process_rss_peak_mb"],
        "python_peak_memory_mb": measurement["python_peak_memory_mb"],
        "process_rss_before_mb": measurement["process_rss_before_mb"],
        "rss_sampling_interval_s": measurement["rss_sampling_interval_s"],
        "rss_sampling_limitation": measurement["rss_sampling_limitation"],
        "prepared_solver_path": PREPARED_SOLVER_PATH,
        "generator_limit_semantics": GENERATOR_LIMIT_SEMANTICS,
        **counts,
        "error_type": error["type"] if error else None,
        "error_message": error["message"] if error else None,
    }


def skipped(horizon, root, repeat_index):
    counts = model_counts(root, horizon, storage_assets=1, owner_groups=1)
    reason = f"storage horizon exceeds STORAGE_MAX_HORIZON={STORAGE_MAX_HORIZON}"
    return {
        "horizon_hours": horizon, "artifact_dir": str(root),
        "variant": "storage_horizon", "storage_variant": "storage",
        "repeat_index": repeat_index, "worker_count": 1,
        "run_status": "skipped", "solver_status": "not_run",
        "wall_time_s": None, "median_hourly_solve_time_s": None,
        "p95_hourly_solve_time_s": None, "peak_memory_mb": None,
        "process_rss_peak_mb": None, "python_peak_memory_mb": None,
        "process_rss_before_mb": None,
        "rss_sampling_interval_s": RSS_SAMPLING_INTERVAL_S,
        "rss_sampling_limitation": "Case was skipped before solver execution.",
        "prepared_solver_path": PREPARED_SOLVER_PATH,
        "generator_limit_semantics": GENERATOR_LIMIT_SEMANTICS,
        **counts, "error_type": "SkippedByConfiguration", "error_message": reason,
    }


def run_case(horizon, variant, repeat_index):
    root = artifact_for(horizon)
    if (
        variant == "storage_horizon"
        and STORAGE_MAX_HORIZON is not None
        and horizon > STORAGE_MAX_HORIZON
    ):
        return skipped(horizon, root, repeat_index)
    storage = variant == "storage_horizon"
    counts = model_counts(root, horizon, storage_assets=int(storage), owner_groups=int(storage))
    if variant == "serial":
        function, workers = lambda: solve_serial(root, horizon, cold=True), 1
    elif variant == "sparse_reuse":
        function, workers = lambda: solve_serial(root, horizon, cold=False), 1
    elif variant == "chunked_parallel":
        function, workers = lambda: solve_chunked_parallel(root, horizon, PARALLEL_WORKERS), PARALLEL_WORKERS
    elif variant == "storage_horizon":
        function, workers = lambda: solve_storage(root, horizon), 1
    else:
        raise ValueError(variant)
    return row_for(horizon, root, variant, repeat_index, workers, counts, timed(function), storage)


def add_repeat_stats(rows):
    groups = {}
    for row in rows:
        if row["run_status"] == "completed" and row["wall_time_s"] is not None:
            groups.setdefault((row["horizon_hours"], row["variant"]), []).append(row["wall_time_s"])
    for row in rows:
        values = groups.get((row["horizon_hours"], row["variant"]), [])
        row["repeat_count_completed"] = len(values)
        row["median_wall_time_s"] = statistics.median(values) if values else None
        row["p95_wall_time_s"] = percentile(values, 95) if values else None
    return rows


def run_benchmark():
    expected_keys = _expected_case_keys()
    existing = {
        row["case_key"]: row
        for row in load_checkpoint()
        if row.get("case_key") in expected_keys
    }
    for horizon in BENCHMARK_HORIZONS:
        repeats = REPEATS_SHORT if horizon <= SHORT_HORIZON_CUTOFF else REPEATS_LONG
        for variant in ("serial", "sparse_reuse", "chunked_parallel", "storage_horizon"):
            for repeat_index in range(1, repeats + 1):
                key = case_key(horizon, variant, repeat_index)
                if key in existing:
                    print(f"resume case={key}")
                    continue
                print(f"running case={key}")
                row = run_case(horizon, variant, repeat_index)
                row["case_key"] = key
                existing[key] = row
                ordered = [existing[item] for item in expected_keys if item in existing]
                write_checkpoint(add_repeat_stats(ordered))
    return add_repeat_stats([existing[key] for key in expected_keys if key in existing])


def recommend(rows):
    candidates = [
        row for row in rows
        if row["horizon_hours"] == 4416
        and row["variant"] == "sparse_reuse"
        and row["run_status"] == "completed"
        and row["solver_status"] == "optimal"
    ]
    if not candidates:
        return {
            "recommendation": "dependency_limited_unverified",
            "reason": (
                "No completed 4416-hour sparse_reuse measurement with "
                "solver_status == optimal exists; performance is unverified."
            ),
        }
    usable = [
        row for row in candidates
        if isinstance(row.get("median_wall_time_s"), (int, float))
        and isinstance(row.get("peak_memory_mb"), (int, float))
    ]
    if not usable:
        return {
            "recommendation": "dependency_limited_unverified",
            "reason": "Optimal solver status exists but performance fields are missing or non-numeric.",
        }
    wall = max(row["median_wall_time_s"] for row in usable)
    memory = max(row["peak_memory_mb"] for row in usable)
    if wall <= WALL_TIME_BUDGET_S and memory <= PEAK_RSS_BUDGET_MB:
        decision = "retain_official_4416_hour_horizon"
        reason = "Measured optimal full-horizon sparse_reuse results are within configured budgets."
    else:
        decision = "screen_or_shrink_locally_only"
        reason = (
            "Measured optimal full-horizon results exceed a configured budget. "
            "Keep full remote instructor evaluation; shorter runs are screening-only."
        )
    return {
        "recommendation": decision, "reason": reason,
        "median_wall_time_s": wall, "peak_memory_mb": memory,
        "wall_time_budget_s": WALL_TIME_BUDGET_S,
        "peak_rss_budget_mb": PEAK_RSS_BUDGET_MB,
    }


In [ ]:
import hashlib
sys.path.insert(0, str(REPO_DIR / "experiments/colab"))
from checkpointing import (
    configuration_signature,
    load_checkpoint as load_checkpoint_rows,
    matching_checkpoint_rows,
)

VARIANTS = ("serial", "sparse_reuse", "chunked_parallel", "storage_horizon")


def _manifest_identity(root):
    identity = {}
    for name in ("manifest.json", "scenario_profile.json", "economic_profile_manifest.json"):
        path = Path(root) / name
        identity[name] = (
            {"path": str(path.resolve()), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
            if path.is_file() else None
        )
    return identity


def _artifact_identity(root):
    root = Path(root)
    return {"path": str(root.resolve()), "manifest_identity": _manifest_identity(root)}


def _configuration():
    return {
        "schema_version": 2,
        "artifact_identity": {str(horizon): _artifact_identity(artifact_for(horizon)) for horizon in BENCHMARK_HORIZONS},
        "horizons": list(BENCHMARK_HORIZONS),
        "storage_policy": {"mode": "explicit_max_horizon_only", "max_horizon": STORAGE_MAX_HORIZON},
        "storage_max_horizon": STORAGE_MAX_HORIZON,
        "variants": list(VARIANTS),
        "repeats_short": REPEATS_SHORT,
        "repeats_long": REPEATS_LONG,
        "parallel_workers": PARALLEL_WORKERS,
        "solver_model_revision": {
            "repository_revision": REPO_REVISION,
            "highspy_version": importlib.metadata.version("highspy"),
            "prepared_solver_path": PREPARED_SOLVER_PATH,
            "generator_limit_semantics": GENERATOR_LIMIT_SEMANTICS,
        },
        "sampler_settings": {
            "rss_sampling_interval_s": RSS_SAMPLING_INTERVAL_S,
            "method": "psutil.Process.memory_info.rss",
        },
        "wall_time_budget_s": WALL_TIME_BUDGET_S,
        "peak_rss_budget_mb": PEAK_RSS_BUDGET_MB,
    }


CONFIGURATION = _configuration()
CONFIGURATION_SIGNATURE = configuration_signature(CONFIGURATION)


def load_checkpoint():
    return load_checkpoint_rows(CHECKPOINT_JSON, CHECKPOINT_CSV)


def write_checkpoint(rows, recommendation=None):
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    payload = {
        "repository_revision": REPO_REVISION,
        "python": sys.version,
        "platform": platform.platform(),
        "highspy_version": importlib.metadata.version("highspy"),
        "configuration": CONFIGURATION,
        "configuration_signature": CONFIGURATION_SIGNATURE,
        "rows": rows,
        "recommendation": recommendation,
    }
    json_tmp = CHECKPOINT_JSON.with_name(f".{CHECKPOINT_JSON.name}.tmp")
    csv_tmp = CHECKPOINT_CSV.with_name(f".{CHECKPOINT_CSV.name}.tmp")
    json_tmp.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    pd.DataFrame(rows).to_csv(csv_tmp, index=False)
    os.replace(json_tmp, CHECKPOINT_JSON)
    os.replace(csv_tmp, CHECKPOINT_CSV)


_legacy_row_for = row_for
_legacy_skipped = skipped


def row_for(horizon, root, variant, repeat_index, workers, counts, measurement, storage):
    row = _legacy_row_for(horizon, root, variant, repeat_index, workers, counts, measurement, storage)
    row["configuration_signature"] = CONFIGURATION_SIGNATURE
    return row


def skipped(horizon, root, repeat_index):
    row = _legacy_skipped(horizon, root, repeat_index)
    row["configuration_signature"] = CONFIGURATION_SIGNATURE
    return row


def run_benchmark():
    expected_keys = _expected_case_keys()
    loaded = load_checkpoint()
    existing = matching_checkpoint_rows(loaded, CONFIGURATION_SIGNATURE, expected_keys)
    stale_count = sum(
        1 for row in loaded
        if row.get("case_key") in expected_keys
        and row.get("configuration_signature") != CONFIGURATION_SIGNATURE
    )
    if stale_count:
        print(f"stale checkpoint rows ignored: {stale_count}")
    for horizon in BENCHMARK_HORIZONS:
        repeats = REPEATS_SHORT if horizon <= SHORT_HORIZON_CUTOFF else REPEATS_LONG
        for variant in VARIANTS:
            for repeat_index in range(1, repeats + 1):
                key = case_key(horizon, variant, repeat_index)
                if key in existing:
                    print(f"resume case={key}")
                    continue
                print(f"running case={key}")
                row = run_case(horizon, variant, repeat_index)
                row["case_key"] = key
                existing[key] = row
                ordered = [existing[item] for item in expected_keys if item in existing]
                write_checkpoint(add_repeat_stats(ordered))
    return add_repeat_stats([existing[key] for key in expected_keys if key in existing])


In [ ]:
results = run_benchmark()
results_frame = pd.DataFrame(results)
recommendation = recommend(results)
write_checkpoint(results, recommendation=recommendation)
display(results_frame[[
    "horizon_hours", "variant", "storage_variant", "repeat_index",
    "run_status", "solver_status", "wall_time_s",
    "median_hourly_solve_time_s", "p95_hourly_solve_time_s",
    "peak_memory_mb", "process_rss_peak_mb", "rss_sampling_interval_s",
    "worker_count", "network_variable_count", "variable_count",
    "constraint_count",
]])
print("wrote:", CHECKPOINT_CSV)
print("wrote:", CHECKPOINT_JSON)
print(json.dumps(recommendation, indent=2))


## Interpretation

The recommendation is generated only from completed 4,416-hour no-storage `sparse_reuse` rows with `solver_status == "optimal"` and numeric sampled RSS/wall-time fields. Missing, skipped, errored, non-optimal, or dependency-limited rows produce `dependency_limited_unverified`, not a performance conclusion. A student-laptop shorter run is a screening recommendation only; it must not change official horizon or evaluator semantics.


In [ ]:
import matplotlib.pyplot as plt

completed = results_frame[results_frame["run_status"] == "completed"]
if completed.empty:
    print("No completed rows to plot.")
else:
    for variant, group in completed.groupby("variant"):
        plt.plot(group["horizon_hours"], group["wall_time_s"], marker="o", label=variant)
    plt.xscale("log")
    plt.yscale("log")
    plt.xlabel("Horizon (hours)")
    plt.ylabel("Wall time (seconds)")
    plt.title("Measured DCOPF benchmark wall time")
    plt.grid(True, which="both", alpha=0.3)
    plt.legend()
    plt.show()